In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Permite importar tu paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))


# 08 · Registro del modelo

> Spec, plan y tareas: `specs/006-registro-modelo/`. Guía: `material-del-curso/kit/guia-validacion-registro-y-app.md`, sección 5.

En la etapa de optimización el **tracking** guardó *ejecuciones*: cada ensayo, con sus parámetros y métricas. Eso
responde "¿qué probamos?". El **registro de modelos** responde otra pregunta: **"¿cuál es el modelo que usamos, y en
qué versión?"**

| | Tracking | Registro |
|---|---|---|
| Guarda | Ejecuciones (muchas, de prueba) | Versiones del modelo (pocas, candidatas a producción) |
| Pregunta | ¿Qué probamos y qué dio? | ¿Qué modelo está en uso y qué versión? |
| Ejemplo | 30 ensayos de la búsqueda | `venta_cruzada_seguros`, la versión que lleva el alias `champion` |

Un **alias** (`champion`) es una etiqueta móvil: quien use el modelo pide "el champion" y no le importa qué número de
versión es. Si mañana hay uno mejor, se mueve el alias y nadie toca el código.

## 0. Conexión con el seguimiento

In [ ]:
from proyecto.tracking.mlflow_utils import configure_mlflow, registered_model_name, set_experiment

with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter("always")
    donde = configure_mlflow()          # 'databricks (token)' o 'local'; nunca imprime el token
for aviso in avisos:
    print("Aviso:", aviso.message)

EXPERIMENTO = "venta-cruzada-registro"
experimento = set_experiment(EXPERIMENTO)   # en Databricks queda bajo /Users/<usuario>/; no se guarda esa ruta
print("MLflow:", donde, "| experimento:", EXPERIMENTO)

> **Dónde se ejecuta:** este notebook se ejecuta **dentro de Databricks**, en el Git folder sincronizado desde GitHub
> (debe decir `databricks (dentro del workspace)`). Si ves `local`, se está ejecutando en un equipo sin conexión: el
> registro funciona igual pero queda en `mlflow_local.db`, y hay que decirlo en el reporte.

## 1. Cargar el modelo final y los resultados de la prueba

Salen del notebook 07: el modelo en `models/final_pipeline.joblib` y las métricas en `reports/07_metricas_test.json`.

In [ ]:
import json
import subprocess
from importlib.metadata import version as installed_version

import joblib
import pandas as pd

raiz = Path.cwd().resolve().parent
modelo_path = raiz / "models" / "final_pipeline.joblib"
info_path = raiz / "reports" / "07_metricas_test.json"
marca_path = raiz / "reports" / "prueba_abierta.txt"
for archivo in (modelo_path, info_path, marca_path):
    if not archivo.exists():
        raise FileNotFoundError(f"Falta {archivo.name}: ejecuta primero el notebook 07 (o actualiza el repo desde GitHub).")

# Un modelo guardado con una versión de scikit-learn/LightGBM puede cargar mal con otra: se exige la de requirements.txt
fijadas = dict(l.strip().split("==") for l in (raiz / "requirements.txt").read_text(encoding="utf-8").splitlines()
               if "==" in l and not l.startswith("#"))
for paquete in ("scikit-learn", "lightgbm"):
    if installed_version(paquete) != fijadas[paquete]:
        raise RuntimeError(f"{paquete} {installed_version(paquete)} instalado, se necesita {fijadas[paquete]}. "
                           "Ejecuta `%pip install -r ../requirements.txt` y reinicia Python.")

modelo = joblib.load(modelo_path)    # el mismo modelo de la etapa 07, sin reentrenar
info = json.loads(info_path.read_text(encoding="utf-8"))
fecha_prueba = next(l.split(":", 1)[1].strip() for l in marca_path.read_text(encoding="utf-8").splitlines()
                    if l.startswith("fecha:"))
try:   # en Databricks el Git folder puede no tener el comando git disponible
    commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=raiz, capture_output=True,
                            text=True, check=True).stdout.strip()
except (OSError, subprocess.CalledProcessError):
    commit = "no disponible"
train = pd.read_parquet(raiz / "data" / "processed" / "train.parquet")   # solo entrenamiento: la prueba no se vuelve a leer
print("modelo y resultados cargados | prueba abierta el", fecha_prueba, "| commit", commit)

## 2. Qué se registra

El modelo final **con lo que hace falta para confiar en él**:

- los **parámetros** y el **tipo de modelo**,
- las **métricas en la prueba** (la evaluación que se hizo una sola vez),
- una **firma** (qué columnas espera y de qué tipo) y un **ejemplo de entrada**,
- **etiquetas** que dicen de dónde salió.

> **Nota técnica.** MLflow guarda por defecto los modelos de scikit-learn en un formato que **rechaza** un `Pipeline` con
> preprocesamiento ("untrusted types"). Por eso el registro usa `serialization_format="cloudpickle"`. Los archivos
> *pickle* **pueden ejecutar código al cargarse**: carga solo modelos que guardaste tú o tu grupo, nunca uno de una
> fuente en la que no confíes.

In [ ]:
import mlflow
from mlflow.models import infer_signature

from proyecto.tracking.registry import max_proba_difference, registry_metrics, registry_params

NOMBRE_MODELO = "venta_cruzada_seguros"  # mismo nombre que ya usa el grupo en Databricks (007)
# Las columnas con que se entrenó el Pipeline, sin `id` (el modelo la descarta: se comprobó que predice igual sin ella)
COLUMNAS_ENTRADA = ["Gender", "Age", "Driving_License", "Region_Code", "Previously_Insured",
                    "Vehicle_Age", "Vehicle_Damage", "Annual_Premium", "Vintage", "is_min_premium"]

nombre_completo = registered_model_name(NOMBRE_MODELO)
ejemplo = train[COLUMNAS_ENTRADA].head(5)
firma = infer_signature(ejemplo, modelo.predict_proba(ejemplo))
etiquetas = {
    "etapa": "08_registro",
    "proyecto": "venta-cruzada",
    "commit": commit,
    "prueba_abierta_el": fecha_prueba,
    "nota": "prueba abierta una sola vez en la etapa 07; métricas copiadas de reports/07_metricas_test.json",
    "mejor_que_linea_base": str(info["prueba"]["mejor_que_linea_base"]),
}

with mlflow.start_run(run_name="modelo-final") as run:
    mlflow.log_params(registry_params(info["decisiones"]))
    mlflow.log_metrics(registry_metrics(info["prueba"]))
    mlflow.set_tags(etiquetas)
    modelo_info = mlflow.sklearn.log_model(
        modelo, name="model", signature=firma, input_example=ejemplo,
        registered_model_name=nombre_completo,
        serialization_format="cloudpickle",          # el formato por defecto rechaza un Pipeline con transformadores propios
        code_paths=[str(raiz / "src" / "proyecto")],  # el Pipeline usa funciones de `proyecto`: sin esto no carga en otro equipo
    )
    run_id = run.info.run_id
version = modelo_info.registered_model_version
print("ejecución:", run_id, "| modelo registrado como:", nombre_completo, "| versión:", version)

## 3. Poner el alias `champion`

Busca la versión recién registrada y dale el alias. A partir de ahora cualquiera pide `models:/<nombre>@champion`.

In [ ]:
cliente = mlflow.MlflowClient()
cliente.set_registered_model_alias(nombre_completo, "champion", str(version))
print(f"{nombre_completo}: versión {version} -> alias 'champion'")

## 4. Comprobación de ida y vuelta

Un registro que no se puede **recuperar** no sirve. Carga el modelo desde el registro y comprueba que predice
**exactamente igual** que el modelo local.

In [ ]:
recuperado = mlflow.sklearn.load_model(f"models:/{nombre_completo}@champion")
muestra = train[COLUMNAS_ENTRADA].head(200)
diferencia = max_proba_difference(modelo, recuperado, muestra)
print("diferencia máxima de probabilidad (200 clientes):", diferencia)
assert diferencia <= 1e-9, "el modelo recuperado del registro no predice igual que el local"
print("el modelo recuperado del registro predice igual que el original")

## 5. Guardar la constancia

Anota dónde quedó el modelo en `reports/08_registro.json` y completa `reports/08_registro.md`: qué versión es la
`champion`, dónde está registrada (Databricks o local) y quién la registró.

In [ ]:
from datetime import datetime

constancia = {
    "tracking": donde,                 # 'databricks (token)' o 'local'; sin dirección del workspace
    "experimento": EXPERIMENTO,        # nombre corto: la ruta completa lleva el correo del usuario
    "modelo_registrado": nombre_completo,
    "version": int(version),
    "alias": "champion",
    "run_id": run_id,
    "columnas_entrada": COLUMNAS_ENTRADA,
    "ida_y_vuelta_diferencia_maxima": diferencia,
    "registrado_el": datetime.now().isoformat(timespec="seconds"),
}
(raiz / "reports" / "08_registro.json").write_text(json.dumps(constancia, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(constancia, indent=2, ensure_ascii=False))

## Qué sigue

El notebook 09 exporta el modelo `champion` a un archivo (`models/model.joblib`) que viaja con la app. **La app no
consulta Databricks**: no se ponen credenciales en la nube, y el despliegue no depende de la cuota de la plataforma.
El registro sigue siendo la **fuente de verdad**: el archivo siempre se exporta desde la versión `champion`.